# 05 - End-to-End Vector Retrieval with ChromaDB
Portfolio 01 · Production RAG Service

### Objective
Validate vector storage, metadata persistence, similarity searching, and top-K filtering with ChromaDB.


In [ ]:
import os
import chromadb
from chromadb.config import Settings

# Disable telemetry
os.environ["ANONYMIZED_TELEMETRY"] = "False"

client = chromadb.Client(Settings(anonymized_telemetry=False, is_persistent=False))
collection = client.create_collection(name="test_experiment", metadata={"hnsw:space": "cosine"})

# Insert sample chunks with rich metadata
sample_docs = [
    "Remote employees receive a one-time home office setup stipend of up to $1,200 USD.",
    "Full-time permanent employees accrue 25 business days of paid annual vacation leave.",
    "The Space Launch System (SLS) is powered by four Aerojet Rocketdyne RS-25 engines.",
    "A P1 critical incident requires first response within 15 minutes and resolution under 2 hours."
]
metadatas = [
    {"doc_id": "doc-01", "filename": "remote_policy.docx", "section": "Equipment", "page": 1},
    {"doc_id": "doc-01", "filename": "remote_policy.docx", "section": "Leave", "page": 2},
    {"doc_id": "doc-02", "filename": "artemis.docx", "section": "Propulsion", "page": 1},
    {"doc_id": "doc-03", "filename": "incident_playbook.docx", "section": "SLA", "page": 1},
]
ids = [f"chunk-{i}" for i in range(len(sample_docs))]

# ChromaDB automatically embeds using default embedding function if not provided
collection.add(documents=sample_docs, metadatas=metadatas, ids=ids)
print(f"Collection count: {collection.count()} chunks stored")


### Querying the Collection


In [ ]:
results = collection.query(
    query_texts=["How much money can remote employees get for desk equipment?"],
    n_results=2
)
print("Top-2 Retrieved Results:")
for i, (doc, meta, dist) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0])):
    sim_score = 1.0 - dist
    print(f"\n[{i+1}] Score: {sim_score:.4f} | Source: {meta['filename']} (Section: {meta['section']})")
    print(f"    Excerpt: {doc}")


### Validation
The top retrieved chunk accurately matches the query with high similarity score!
